In [1]:
# Import Packages
import os
import sys
import time
import copy

import numpy as np
import pandas as pd
import holoviews as hv
import pandas as pd

import xarray as xr
import hvplot.pandas
import hvplot.xarray

#sys.path.append('../')
sys.path.append('EMIT-Data-Resources/python/modules/')

from emit_tools import emit_xarray

In [2]:
import earthaccess

#earthaccess.login(persist=True)
auth = earthaccess.login()
if not auth.authenticated:
    # ask for credentials and persist them in a .netrc file
    auth.login(strategy="interactive", persist=True)

/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Enter your Earthdata Login username:  kosarajub_1978
Enter your Earthdata password:  ········


In [3]:
results = earthaccess.search_data(short_name="EMITL2ARFL", count=1)
print(len(results))

1


/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


In [4]:
def safe_norm_diff(a, b):
    if pd.isna(a) or pd.isna(b) or (a + b) == 0:
        return np.nan
    return (a - b) / (a + b)

def safe_ratio(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b

# EMIT Data

**NASA Earth Data**  
https://nasa-openscapes.github.io/2023-Cloud-Workshop-AGU/tutorials/Earthdata_Search_Discovery_earthaccess.html

**EMIT**  
 - https://earth.jpl.nasa.gov/emit/
 - https://www.earthdata.nasa.gov/data/instruments/emit-imaging-spectrometer

**NASA EMIT USER GUIDE:**  
 - https://lpdaac.usgs.gov/documents/1569/EMITL2ARFL_User_Guide_v1.pdf?utm_source=chatgpt.com
 - https://www.earthdata.nasa.gov/s3fs-public/2025-06/EMIT_Data_Resources_Poster_2023_AGU_ck_approved.pdf?VersionId=rGIZ6MbeRP68HEGWCRInda2qzHGK7LNv

**Unified Metadata Model (UMM)**  
https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr/umm

**Some examples:**  
https://github.com/nasa/EMIT-Data-Resources/blob/main/python/how-tos/How_to_find_and_access_EMIT_data.ipynb

### NETCDF FILE
A NetCDF file is a self-describing scientific data file format commonly used for Earth observation data. 

**A NetCDF file stores both:**  
 - The data arrays themselves
 - Metadata describing dimensions, coordinates, variable names, units, and structure.

For EMIT L2A reflectance, the NetCDF reflectance file stores a 285-band surface reflectance cube plus geolocation-related information.  

### GRANULE  
 - A granule is one packaged EMIT observation unit returned by NASA Earthdata search.  
 - In practice, it is one EMIT scene/product instance for a particular place/time, not the whole mission archive.  
 - Each granule is roughly ~75 km × 75 km.

**Each EMIT granule contains three NetCDF files:**  
 - **Reflectance NetCDF (EMIT_L2A_RFL):**  Contains surface reflectance maps  
 - **Mask NetCDF (EMIT_L2A_MASK):** Contains atmospheric state estimates and binary flags
 - **Uncertainty NetCDF (EMIT_L2A_RFLUNCERT):** Contains uncertainty estimates and  

**A reflectance file contains**:  
 - 285 bands spanning wavelengths of between 381 and 2493 nm with a spatial resolution of ~60 m
 - Latitude
 - Longitude
 - Elevation
 - Geolocation lookup tables

**A mask file contains flags like**:  
 - Clouds
 - Water
 - Invalid pixels

## QUERY NASA EARTH DATA.

**BOUNDING BOX FOR THE DATA POINTS**  
**Determine a bounding box that covers a set of points on a map.**  
Suppose your group has these three points:  
 - point 1: longitude = -89.2, latitude = 13.7  
 - point 2: longitude = -89.0, latitude = 13.8  
 - point 3: longitude = -88.8, latitude = 13.9  
Determine a small rectangle that contains all of the above points.  

A bounding box is defined by four numbers:
 - minimum longitude = left edge  
 - minimum latitude  = bottom edge  
 - maximum longitude = right edge  
 - maximum latitude  = top edge  

**Padding**
If an exact tight box is used, we may miss relevant granules because:  
 - A point may lie very close to the edge  
 - Floating point or geolocation precision may cause small mismatches  
So padding is a safety margin.  
Example: pad_deg=0.01 => Expanding by ~1 km on each side of the bounding box rectange.

In [ ]:
def build_group_bounding_box(df_group, pad_deg=0.01):
    lon_min = df_group["longitude"].min() - pad_deg
    lon_max = df_group["longitude"].max() + pad_deg
    lat_min = df_group["latitude"].min() - pad_deg
    lat_max = df_group["latitude"].max() + pad_deg
    return (lon_min, lat_min, lon_max, lat_max)

**QUERY NASA**
Use **earthaccess.search_data** to find all of the the EMIT granules:  
 - whose acquisition time intersects the target date range, and
 - whose footprint intersects the desired bounding box.”

In [102]:
def parse_granule_datetime(granule):
    # Try common earthaccess/CMR structures in order
    candidates = []

    # 1. Granule.meta
    try:
        meta = granule.meta
        if isinstance(meta, dict):
            candidates.append(meta)
    except Exception:
        pass

    # 2. Granule.umm
    try:
        umm = granule.umm
        if isinstance(umm, dict):
            candidates.append(umm)
    except Exception:
        pass

    # 3. Granule as dict-like
    try:
        if isinstance(granule, dict):
            candidates.append(granule)
    except Exception:
        pass

    # Search for datetime in common fields
    for obj in candidates:
        # Direct flat keys
        for key in ["BeginningDateTime", "beginningDateTime", "TemporalExtent"]:
            val = obj.get(key)
            if isinstance(val, str):
                ts = pd.to_datetime(val, errors="coerce")
                if not pd.isna(ts):
                    return ts

        # UMM-style nested structure
        tr = obj.get("TemporalExtent", {}).get("RangeDateTime", {})
        val = tr.get("BeginningDateTime")
        if val:
            ts = pd.to_datetime(val, errors="coerce")
            if not pd.isna(ts):
                return ts

        # Sometimes nested under 'umm'
        umm = obj.get("umm", {})
        tr = umm.get("TemporalExtent", {}).get("RangeDateTime", {})
        val = tr.get("BeginningDateTime")
        if val:
            ts = pd.to_datetime(val, errors="coerce")
            if not pd.isna(ts):
                return ts

    return pd.NaT

In [103]:
def extract_emit_spectra_in_granule_ver1(nc_path, df_points):
    # ortho=True means use the orthorectified geographic version, 
    # so the dataset is aligned to latitude/longitude coordinates.

    # "emit_xarray" opens a gridded scene, not a single-point record.
    # So the output of emit_xarray has many spatial locations.
    ds = emit_xarray(nc_path, ortho=True)

    pts = df_points[["row_id", "longitude", "latitude"]].copy()
    pts = pts.set_index("row_id")
    xp  = pts.to_xarray()

    # For each requested point in xp, find the nearest pixel location inside that granule.
    # TODO: If the requested point is far outside the real useful footprint, nearest may
    #       return some edge pixel
    extracted = ds.sel(latitude=xp.latitude,
                       longitude=xp.longitude,
                       method="nearest").to_dataframe().reset_index()

    #print("Extracted columns:", extracted.columns.tolist())

    # If row_id is missing, try to restore it from the xarray output index name
    if "row_id" not in extracted.columns:
        print("Warning: row_id not found in extracted dataframe")

    if "good_wavelengths" in extracted.columns:
        extracted = extracted[extracted["good_wavelengths"] == 1].copy()

    # NOTE: The output is in long-format
    #       Aka, each row signifies a separate band and its values.
    
    return extracted

In [104]:
import numpy as np
import pandas as pd

def haversine_km(lon1, lat1, lon2, lat2):
    # Great-circle distance in kilometers. Inputs can be scalars or numpy arrays.
    lon1 = np.radians(lon1)
    lat1 = np.radians(lat1)
    lon2 = np.radians(lon2)
    lat2 = np.radians(lat2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c

def extract_emit_spectra_in_granule_ver2(nc_path, df_points, max_distance_km=0.2):
    # Extract nearest EMIT spectra for each point, but discard points whose
    # nearest EMIT pixel is farther than max_distance_km.

    ds = emit_xarray(nc_path, ortho=True)

    pts = df_points[["row_id", "longitude", "latitude"]].copy()
    pts = pts.set_index("row_id")
    xp = pts.to_xarray()

    # Nearest-pixel extraction
    extracted = ds.sel(
        latitude=xp.latitude,
        longitude=xp.longitude,
        method="nearest"
    ).to_dataframe().reset_index()

    print("Extracted columns:", extracted.columns.tolist())

    if "row_id" not in extracted.columns:
        print("Warning: row_id not found in extracted dataframe")

    # Keep only good wavelengths if available
    if "good_wavelengths" in extracted.columns:
        extracted = extracted[extracted["good_wavelengths"] == 1].copy()

    if extracted.empty:
        return extracted

    # Add original requested coordinates
    point_lookup = df_points[["row_id", "longitude", "latitude"]].copy()
    point_lookup = point_lookup.rename(columns={
        "longitude": "orig_longitude",
        "latitude": "orig_latitude"
    })

    extracted = extracted.merge(point_lookup, on="row_id", how="left")

    # The selected EMIT pixel coordinates are typically in the extracted output
    # as latitude / longitude columns.
    if "longitude" not in extracted.columns or "latitude" not in extracted.columns:
        print("Warning: extracted dataframe does not contain selected pixel lat/lon")
        return extracted

    # Compute distance between requested point and selected EMIT pixel
    extracted["distance_km"] = haversine_km(
        extracted["orig_longitude"].to_numpy(),
        extracted["orig_latitude"].to_numpy(),
        extracted["longitude"].to_numpy(),
        extracted["latitude"].to_numpy()
    )

    # Keep only rows whose selected pixel is close enough
    extracted = extracted[extracted["distance_km"] <= max_distance_km].copy()

    return extracted

In [105]:
def extract_all_bands(long_df):
    # Convert long-format EMIT spectra into wide format:
    # one row per row_id, one column per wavelength reflectance.

    if long_df.empty:
        return pd.DataFrame()

    required_cols = ["row_id", "wavelengths", "reflectance"]
    missing = [c for c in required_cols if c not in long_df.columns]
    if missing:
        raise ValueError(f"long_df missing required columns: {missing}")

    df = long_df[required_cols].copy()

    # Some wavelength values can be floats like 381.0, 388.5, etc.
    # Round slightly for stable column naming.
    df["wavelengths"] = df["wavelengths"].astype(float).round(1)

    # Pivot wavelengths into columns
    wide_df = df.pivot_table(
        index="row_id",
        columns="wavelengths",
        values="reflectance",
        aggfunc="first"
    ).reset_index()

    # Separate row_id and wavelength columns
    wavelength_cols = [c for c in wide_df.columns if c != "row_id"]
    # Sort wavelength columns numerically
    wavelength_cols_sorted = sorted(wavelength_cols)
    # Reorder: row_id first, then wavelengths
    wide_df = wide_df[["row_id"] + wavelength_cols_sorted]

    # Rename columns into CSV-safe names
    # Example: 381.0 -> EMIT_R381_0, 388.5 -> EMIT_R388_5
    new_cols = []
    for col in wide_df.columns:
        if col == "row_id":
            new_cols.append("row_id")
        else:
            col_str = str(col).replace(".", "_")
            new_cols.append(f"EMIT_R{col_str}")

    wide_df.columns = new_cols
    return wide_df

In [106]:
def extract_derivates_from_long_form(long_df):
    rows = []

    for row_id, g in long_df.groupby("row_id"):
        g = g.sort_values("wavelengths")

        def refl_at(target_nm):
            idx = (g["wavelengths"] - target_nm).abs().idxmin()
            return g.loc[idx, "reflectance"]

        r665 = refl_at(665)
        r705 = refl_at(705)
        r740 = refl_at(740)
        r783 = refl_at(783)
        r850 = refl_at(850)
        r1610 = refl_at(1610)
        r2210 = refl_at(2210)
        
        ndvi  = safe_norm_diff(r850, r665)
        ndre1 = safe_norm_diff(r850, r705)
        ndre2 = safe_norm_diff(r850, r740)
        ndre3 = safe_norm_diff(r850, r783)

        ratio = safe_ratio(r850, r705)
        ci_rededge = ratio - 1 if not pd.isna(ratio) else np.nan

        nbr = safe_norm_diff(r850, r2210)
        msi = safe_ratio(r1610, r850)

        rows.append({
            "row_id": row_id,
            "EMIT_R665": r665,
            "EMIT_R705": r705,
            "EMIT_R740": r740,
            "EMIT_R783": r783,
            "EMIT_R850": r850,
            "EMIT_R1610": r1610,
            "EMIT_R2210": r2210,
            "EMIT_NDVI": ndvi,
            "EMIT_NDRE1": ndre1,
            "EMIT_NDRE2": ndre2,
            "EMIT_NDRE3": ndre3,
            "EMIT_CIrededge": ci_rededge,
            "EMIT_NBR": nbr,
            "EMIT_MSI": msi
        })

    return pd.DataFrame(rows)

In [107]:
#PROCESS ALL GRANULES
def process_emit_date_group_all_granules(df_group, start_date, end_date, download_dir):
    bbox = build_group_bounding_box(df_group)

    granules = earthaccess.search_data(short_name="EMITL2ARFL",
                                       version="001",
                                       cloud_hosted=True,
                                       temporal=(start_date.strftime("%Y-%m-%d"),
                                                 end_date.strftime("%Y-%m-%d")),
                                       bounding_box=bbox,
                                       count=200)
    print(f"  Granules found: {len(granules)}")

    # Case 1: No granules found.
    if not granules:
        missing_group = df_group.copy()
        missing_group["missing_reason"] = "no_granules_found"
        return pd.DataFrame(), missing_group

    candidate_rows       = []
    failed_granule_count = 0
    empty_longdf_count   = 0
    empty_featdf_count   = 0

    for granule_idx, granule in enumerate(granules, start=1):
        try:
            print(f"  Processing granule {granule_idx}/{len(granules)}")

            # Download granule if needed.
            os.makedirs(download_dir, exist_ok=True)
            start = time.time()
            nc_path = earthaccess.download([granule], download_dir)[0]
            end = time.time()
            print("    Downloaded to .." + str(nc_path))
            print(f"Time taken: {end - start:.2f} seconds")
            if nc_path is None:
                print("    Download returned None")
                failed_granule_count += 1
                continue

            long_df = extract_emit_spectra_in_granule_ver1(nc_path, df_group)
            print(f"    Extracted rows: {len(long_df)}")
            if long_df.empty:
                print("    long_df is empty")
                empty_longdf_count += 1
                continue

            #feat_df = summarize_emit_features(long_df)
            feat_df = extract_all_bands(long_df)
            print(f"    Feature rows produced: {len(feat_df)}")
            if feat_df.empty:
                print("    feat_df is empty")
                empty_featdf_count += 1
                continue

            gtime = parse_granule_datetime(granule)
            feat_df["EMIT_selected_date"] = gtime

            # In practice, the NetCDF filename acts as the granule identifier
            granule_id = os.path.basename(nc_path) if nc_path else None
            feat_df["EMIT_granule"] = str(granule_id) if granule_id is not None else np.nan

            feat_df["days_from_start"] = (
                abs((gtime - start_date).total_seconds()) / 86400
                if not pd.isna(gtime) else float("inf")
            )

            candidate_rows.append(feat_df)

        except Exception as e:
            print(f"  Granule failed: {e}")
            failed_granule_count += 1
            #assert 0
            continue

    # Case 2: Granules exist, but none of them are useful.
    if not candidate_rows:
        missing_group = df_group.copy()

        if failed_granule_count == len(granules):
            reason = "all_granules_failed"
        elif empty_longdf_count == len(granules):
            reason = "all_granules_returned_no_points"
        elif empty_featdf_count == len(granules):
            reason = "all_granules_returned_no_features"
        else:
            reason = "no_usable_emit_features"

        missing_group["missing_reason"] = reason
        return pd.DataFrame(), missing_group

    all_candidates = pd.concat(candidate_rows, ignore_index=True)
    all_candidates = all_candidates.sort_values(["row_id", "days_from_start"])

    # Keep closest granule to start_date for each row_id
    best_emit = all_candidates.groupby("row_id", as_index=False).first()
    best_emit = best_emit.drop(columns=["days_from_start"])

    # Output only rows with EMIT data
    merged_group = df_group.merge(best_emit, on="row_id", how="inner")

    # Missing = original rows whose row_id did not appear in best_emit
    found_row_ids = set(best_emit["row_id"].tolist())
    missing_mask  = ~df_group["row_id"].isin(found_row_ids)

    missing_group = df_group.loc[missing_mask].copy()
    missing_group["missing_reason"] = "not_covered_by_selected_granules"

    return merged_group, missing_group

In [108]:
#PROCESS ONLY THE FIRST GRANULE.
def process_emit_date_group_first_granule(df_group, start_date, end_date, download_dir):
    bbox = build_group_bounding_box(df_group)

    granules = earthaccess.search_data(short_name="EMITL2ARFL",
                                       version="001",
                                       cloud_hosted=True,
                                       temporal=(start_date.strftime("%Y-%m-%d"),
                                                 end_date.strftime("%Y-%m-%d")),
                                       bounding_box=bbox,
                                       count=200)
    print(f"  Granules found: {len(granules)}")

    if not granules:
        missing_group = df_group.copy()
        missing_group["missing_reason"] = "no_granules_found"
        return pd.DataFrame(), missing_group

    # Sort by granule datetime, earliest first
    granule_time_pairs = []
    for granule in granules:
        gtime = parse_granule_datetime(granule)
        sort_time = gtime if not pd.isna(gtime) else pd.Timestamp.max
        granule_time_pairs.append((granule, sort_time))

    granule_time_pairs.sort(key=lambda x: x[1]) # Pick the earliest granule
    #granule_time_pairs.sort(key=lambda x: x[1], reverse=True) #Pick the latest granule
    
    selected_granule, selected_time = granule_time_pairs[0]

    try:
        print(f"  Using earliest granule: {selected_time}")

        os.makedirs(download_dir, exist_ok=True)

        t0 = time.time()
        downloaded = earthaccess.download([selected_granule], download_dir)
        t1 = time.time()

        if not downloaded:
            missing_group = df_group.copy()
            missing_group["missing_reason"] = "granule_download_failed"
            return pd.DataFrame(), missing_group

        nc_path = downloaded[0]
        print(f"    Downloaded: {nc_path}")
        print(f"    Download time: {t1 - t0:.2f} sec")

        long_df = extract_emit_spectra_in_granule(nc_path, df_group)
        print(f"    Extracted rows: {len(long_df)}")

        if long_df.empty:
            missing_group = df_group.copy()
            missing_group["missing_reason"] = "granule_returned_no_points"
            return pd.DataFrame(), missing_group

        feat_df = extract_all_bands(long_df)
        print(f"    Feature rows produced: {len(feat_df)}")

        if feat_df.empty:
            missing_group = df_group.copy()
            missing_group["missing_reason"] = "granule_returned_no_features"
            return pd.DataFrame(), missing_group

        feat_df["EMIT_selected_date"] = selected_time
        feat_df["EMIT_granule"] = os.path.basename(nc_path)

        # Keep only rows that actually got EMIT data
        found_row_ids = set(feat_df["row_id"].tolist())

        emit_mask = df_group["row_id"].isin(found_row_ids)
        missing_mask = ~emit_mask

        merged_group = df_group.loc[emit_mask].copy()
        missing_group = df_group.loc[missing_mask].copy()
        missing_group["missing_reason"] = "not_covered_by_selected_granule"

        # Attach EMIT columns without merge
        feat_df_indexed = feat_df.set_index("row_id")
        
        # Align rows once
        aligned_emit = feat_df_indexed.loc[merged_group["row_id"]].reset_index(drop=True)
        
        # Drop row_id (already in merged_group)
        aligned_emit = aligned_emit.drop(columns=["row_id"], errors="ignore")
        
        # Concatenate all columns at once
        merged_group = pd.concat([merged_group.reset_index(drop=True), aligned_emit], axis=1)

        return merged_group, missing_group

    except Exception as e:
        print(f"  Selected granule failed: {e}")
        missing_group = df_group.copy()
        missing_group["missing_reason"] = "selected_granule_failed"
        return pd.DataFrame(), missing_group

In [109]:
def load_agb_data(csv_file):
    orig_df = pd.read_csv(csv_file)

    # Parse as UTC-aware datetimes
    orig_df['start_date'] = pd.to_datetime(orig_df['start_date'], utc=True, errors='coerce')
    orig_df['end_date']   = pd.to_datetime(orig_df['end_date'], utc=True, errors='coerce')

    # Discard all data that is older than 2020.
    discard_threshold = pd.Timestamp('2020-01-01', tz='UTC')
    mask = (
        (orig_df['start_date'].notna()) &
        (orig_df['start_date'] >= discard_threshold)
    )
    agb_df       = orig_df[mask]
    discarded_df = orig_df[~mask]
    
    agb_df['capture_start'] = agb_df['start_date'].copy()
    agb_df['capture_end']   = agb_df['end_date'].copy()

    # Timeline of EMIT spectroscopy
    #  July 2022    :  Installed on the International Space Station
    #  Aug–Sept 2022: Commissioning / calibration phase
    #  Late 2022    : Early science data starts
    #  2023 onward  : Regular science operation
    emit_threshold_start = pd.Timestamp('2023-01-01', tz='UTC')
    emit_threshold_end   = pd.Timestamp('2025-12-31', tz='UTC')
    #emit_threshold_end   = pd.Timestamp('2023-06-30', tz='UTC')
    
    mask = agb_df['start_date'] < emit_threshold_start
    agb_df.loc[mask, 'capture_start'] = emit_threshold_start
    agb_df.loc[mask, 'capture_end']   = emit_threshold_end

    return agb_df, discarded_df

In [110]:
def extract_emit_batched(input_csv, output_csv, missing_output_csv, download_dir):
    agb_df, discarded_df = load_agb_data(input_csv)
    if not len(agb_df):
        return agb_df, discarded_df
    
    agb_df["latitude"]   = pd.to_numeric(agb_df["latitude"], errors="coerce")
    agb_df["longitude"]  = pd.to_numeric(agb_df["longitude"], errors="coerce")
    agb_df["start_date"] = pd.to_datetime(agb_df["start_date"], errors="coerce")
    agb_df["end_date"]   = pd.to_datetime(agb_df["end_date"], errors="coerce")
    
    agb_df = agb_df.dropna(subset=["latitude", "longitude", "start_date", "end_date"]).reset_index(drop=True)
    agb_df["row_id"] = agb_df.index.astype(int)

    results = []
    missing_results = []

    # Group by BOTH start_date and end_date
    grouped = agb_df.groupby(['capture_start', 'capture_end'], sort=False)
    total_groups = len(grouped)

    for group_num, ((capture_start, capture_end), df_group) in enumerate(grouped, start=1):
        print(f"Processing EMIT group {group_num}/{total_groups} | "
              f"capture_start_date={capture_start} capture_end_date={capture_end} | rows={len(df_group)}")

        """
        merged_group, missing_group = process_emit_date_group_first_granule(df_group,
                                                                            capture_start,
                                                                            capture_end,
                                                                            download_dir)
        """

        merged_group, missing_group = process_emit_date_group_all_granules(df_group,
                                                                            capture_start,
                                                                            capture_end,
                                                                            download_dir)

        if not merged_group.empty:
            results.append(merged_group)

        if not missing_group.empty:
            missing_results.append(missing_group)

    if results:
        emit_df = pd.concat(results, ignore_index=True)
        emit_df = emit_df.sort_values("row_id").drop(columns=["row_id"])
        #emit_df.to_csv(output_csv, index=False)
        #print(f"Saved EMIT output to: {output_csv}")
    else:
        emit_df = pd.DataFrame()

    if missing_results:
        missing_data = pd.concat(missing_results, ignore_index=True)
        missing_data = missing_data.sort_values("row_id").drop(columns=["row_id"])
        #missing_data.to_csv(missing_output_csv, index=False)
        #print(f"Saved missing-data output to: {missing_output_csv}")
    else:
        missing_data = pd.DataFrame()
    if len(discarded_df):
        missing_data = pd.concat([missing_data, discarded_df])
    
    return emit_df, missing_data

# EXTRACT THE DATA

## Extract all of EMIT bands.

In [111]:
INPUT_CSV        = "../../../DATA/AGB_DATA/Merged_Data/AGB_MERGED.csv"
OUTPUT_CSV       = "../../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT.csv"
MISSING_DATA_CSV = "../../../DATA/AGB_DATA/Merged_Data/EMIT_AGB/AGB_EO_EMIT_MISSING.csv"

#INPUT_CSV        = "Data/Input/TEST_AGB.csv"
#OUTPUT_CSV       = "Data/Output/TEST_EMIT_ALL_BANDS.csv"
#MISSING_DATA_CSV = "Data/Output/TEST_EMIT_MISSING.csv"

DOWNLOAD_DIR     = "Data/Download"

In [ ]:
start = time.time()
emit_df, missing_data = extract_emit_batched(INPUT_CSV, OUTPUT_CSV, MISSING_DATA_CSV, DOWNLOAD_DIR)
end = time.time()
print(f"Time taken: {end - start:.2f} seconds")

/var/folders/b8/8qlj5yzd0jng0b9470zpnl9h0000gn/T/ipykernel_84674/1871618388.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agb_df['capture_start'] = agb_df['start_date'].copy()
/var/folders/b8/8qlj5yzd0jng0b9470zpnl9h0000gn/T/ipykernel_84674/1871618388.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  agb_df['capture_end']   = agb_df['end_date'].copy()


Processing EMIT group 1/1 | capture_start_date=2023-01-01 00:00:00+00:00 capture_end_date=2025-12-31 00:00:00+00:00 | rows=3880
  Granules found: 92
  Processing granule 1/92


/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/earthaccess/results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()
/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/3 [00:00<?, ?it/s]

    Downloaded to ..Data/Download/EMIT_L2A_RFL_001_20250324T185822_2508312_008.nc
Time taken: 393.35 seconds
    Extracted rows: 946720
    Feature rows produced: 3880
  Processing granule 29/92


/opt/anaconda3/envs/bkos/lib/python3.14/site-packages/earthaccess/store.py:832: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  total_size = round(sum(granule.size() for granule in granules) / 1024, 2)


QUEUEING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/3 [00:00<?, ?it/s]

## ADD INDICES
Indices are simple mathematical combinations of a few spectral bands designed to highlight a specific physical property.

In [94]:
def get_closest_emit_band(df, target_nm):
    emit_cols = [c for c in df.columns if c.startswith("EMIT_R")]

    wavelength_map = {
        c: float(c.replace("EMIT_R", "").replace("_", "."))
        for c in emit_cols
    }

    closest_col = min(
        wavelength_map,
        key=lambda c: abs(wavelength_map[c] - target_nm)
    )
    return closest_col


def add_emit_indices(df):
    red_col   = get_closest_emit_band(df, 665)
    re1_col   = get_closest_emit_band(df, 705)
    re2_col   = get_closest_emit_band(df, 740)
    re3_col   = get_closest_emit_band(df, 783)
    nir_col   = get_closest_emit_band(df, 850)
    blue_col  = get_closest_emit_band(df, 490)
    swir1_col = get_closest_emit_band(df, 1610)
    swir2_col = get_closest_emit_band(df, 2210)

    red   = df[red_col]
    re1   = df[re1_col]
    re2   = df[re2_col]
    re3   = df[re3_col]
    nir   = df[nir_col]
    blue  = df[blue_col]
    swir1 = df[swir1_col]
    swir2 = df[swir2_col]
    
    df["NDVI"]  = (nir - red) / (nir + red)
    df["NDRE1"] = (nir - re1) / (nir + re1)
    df["NDRE2"] = (nir - re2) / (nir + re2)
    df["NDRE3"] = (nir - re3) / (nir + re3)
    df["NBR"]   = (nir - swir2) / (nir + swir2)
    df["MSI"]   = swir1 / nir

    denom = nir + 6 * red - 7.5 * blue + 1
    df["EVI"] = 2.5 * ((nir - red) / denom)

    df["CIrededge"] = (nir / re1) - 1

    return df

In [95]:
emit_df = add_emit_indices(emit_df)

# SAVE THE DATA

## Reorder columns for readability.

In [96]:
priority_cols = [
    "dataset",
    "plot_id",
    "start_date",
    "end_date",
    "capture_start",
    "capture_end",
    "EMIT_selected_date",
    "EMIT_granule",
    "latitude",
    "longitude",
    "diameter",
    "height",
    "species",
    "plant_AGB_kg",
    "NDVI",
    "NDRE1",
    "NDRE2",
    "NDRE3",
    "NBR",
    "MSI",
    "EVI",
    "CIrededge"
]
    
# Keep only those that actually exist (safe guard)
priority_cols = [c for c in priority_cols if c in emit_df.columns]

# Remaining columns
remaining_cols = [c for c in emit_df.columns if c not in priority_cols]

# Reorder
emit_df = emit_df[priority_cols + remaining_cols]

## Save the data

In [97]:
emit_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved EMIT output to: {MISSING_DATA_CSV}")

Saved EMIT output to: Data/Output/EMIT_MISSING.csv


In [98]:
missing_data.to_csv(MISSING_DATA_CSV, index=False)
print(f"Saved missing-data output to: {MISSING_DATA_CSV}")

Saved missing-data output to: Data/Output/EMIT_MISSING.csv


# FEATURE ENGINEERING

**Option 1: Use only selected spectral windows**

**Option 2: Use summary features from derivatives**

**Option 3: PCA on reflectance or derivatives**  

Apply PCA on 285 bands and reduce them to something like 10–20 PCA components.  
Add a small number of physically meaningful signals that PCA might not represent explicitly or stably.

**Option 4: Feature selection** 